# rPPG · 实验笔记本（可见版）

这个笔记本**不打包**任何东西：一格只做一件事，每格开头写清楚「做什么 / 为什么 / 上一格的产物是什么」。
上游 `rPPG-Toolbox` 一行不改，需要对照的地方直接引用它的行号。

它要回答的问题只有一个：**预处理为什么让 GPU 干等，怎么改才能快。**

| 格 | 做什么 | 产物 |
|---|---|---|
| 0 | 环境（原样，不改） | `ctx` |
| 1 | 参数 + 计时工具 | `tick` / `TIMES` |
| 2 | 看数据 + 量读盘速度 | 单视频大小、MB/s |
| 3 | 基线：上游逐帧流程（分阶段计时） | 耗时明细 |
| 4 | 快版：ffmpeg 一次 decode+crop+scale | 72×72 数组 |
| 5 | 两条路的数值差 | 差值统计 |
| 6 | 全量预处理 → 本地缓存（uint8 + 统计量，数据集指纹） | `_cache/<fp>/` |
| 7 | 缓存打包上传 Drive | `<fp>.tar.gz` + `<fp>.json` |
| 8 | 新 VM 解包（有 tar 就跳过预处理） | 本地缓存 |
| 9 | 边产边推：CPU 预处理与 GPU 推理重叠 | chunk 指标 |
| 10 | 汇总 | 表 |


In [ ]:
# ============ 第 0 格：环境（固定块，永远不要改）============
# 幂等：挂 Drive → 拉代码 → 判新 VM → 装依赖/打补丁 → 载凭据 → 备数据
# 如果这一格只转圈、一个字都不打印：内核被上一轮卡死了 → 先 Restart Kernel
print("[0/3] 同步代码仓库…", flush=True)

import os, subprocess, sys
from pathlib import Path

os.environ.setdefault("GIT_TERMINAL_PROMPT", "0")   # git 绝不弹账号密码，宁可报错
repo = Path("/content/colab-workbench")
try:
    if not repo.exists():
        subprocess.run(["git", "clone", "-q",
            "https://github.com/Elisabethhui/colab-workbench.git",
            str(repo)], check=True, timeout=180)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "-q", "--ff-only"],
                       check=False, timeout=180)
except subprocess.TimeoutExpired:
    raise RuntimeError("git 同步超过 180 秒，网络可能不通。先确认 VM 能访问 github.com") from None
head = subprocess.run(["git", "-C", str(repo), "log", "--oneline", "-1"],
                      capture_output=True, text=True).stdout.strip()
print("[1/3] 代码版本:", head, flush=True)

# 同一个内核里重跑时，必须丢掉旧的模块缓存，否则改动不生效
for _m in [m for m in sys.modules if m == "bootstrap" or m.startswith("projects")]:
    del sys.modules[_m]
sys.path.insert(0, str(repo))

print("[2/3] 进入 bootstrap.setup()（下一步是挂 Drive，会弹授权）", flush=True)
from bootstrap import setup
ctx = setup("rppg")
print("[3/3] 完成", flush=True)


## 第 1 格 · 参数与计时工具

后面每一格都用 `with tick("名字"):` 包住要计时的代码，耗时自动累加。
**只改这一格**就能换数据集、换分辨率、换跑多少个 subject。


In [ ]:
# 第 1 格：参数 + 计时工具
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
import cv2
import numpy as np
import torch

DATASET_VIEW = Path("/content/UBFC")       # 第 0 格建好的软链视图（42 个 subject）
REPO = Path(ctx.paths.repo)                 # 上游 rPPG-Toolbox（已钉死版本）
CACHE_LOCAL = Path("/content/_cache")       # 缓存写本地盘：开 FUSE 写 Drive 会慢一个量级
CACHE_LOCAL.mkdir(parents=True, exist_ok=True)

PREPROC_VERSION = 2     # 预处理代码的版本号：改了 crop/scale 逻辑必须 +1，旧缓存自动失效
W = H = 72          # = PREPROCESS.RESIZE.W/H
CHUNK = 180         # = CHUNK_LENGTH（30fps × 6s）
LARGE_COEF = 1.5    # = CROP_FACE.LARGE_BOX_COEF
FRAME_DEPTH = 10    # = MODEL.EFFICIENTPHYS.FRAME_DEPTH（TSM 段长，帧数必须能被它整除）
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SUBSET = None       # None = 全量 42 个；调试时改成 3 只跑前 3 个

def face_box(frame):
    """= BaseLoader.face_detection(backend='HC') + use_larger_box。
    frame 要求 RGB（上游就是在 RGB 上检测的）。整段视频只调一次（DO_DYNAMIC_DETECTION=False）。
    """
    det = cv2.CascadeClassifier(str(REPO / "dataset/haarcascade_frontalface_default.xml"))
    zone = det.detectMultiScale(frame[:, :, :3].astype(np.uint8))
    if len(zone) < 1:
        print("    (未检出人脸 → 上游会退回整帧)")
        box = [0, 0, frame.shape[1], frame.shape[0]]
    else:
        box = [int(v) for v in zone[np.argmax(zone[:, 2])]]
    box[0] = max(0, box[0] - (LARGE_COEF - 1.0) / 2 * box[2])
    box[1] = max(0, box[1] - (LARGE_COEF - 1.0) / 2 * box[3])
    box[2] = LARGE_COEF * box[2]
    box[3] = LARGE_COEF * box[3]
    return [int(v) for v in box]


def clamp_box(box, frame_hw):
    """把框裁进画面内，与上游 numpy 切片的越界行为一致。

    上游 crop_face_resize 是 `f[y:y+bh, x:x+bw]`，越界部分 numpy 会自动截断；
    ffmpeg 的 crop 越界时自己另有一套处理，两边会算出不同的画面
    （实测 subject1 的框 y+bh=492 > 480，平均差 18 个灰度级）。
    所以这里先按上游语义截断，再交给 ffmpeg。
    """
    H0, W0 = frame_hw
    x, y, bw, bh = [int(round(v)) for v in box]
    x = max(0, min(x, W0 - 2))
    y = max(0, min(y, H0 - 2))
    bw = max(2, min(bw, W0 - x))
    bh = max(2, min(bh, H0 - y))
    return [x, y, bw, bh]       # 不做"偶数化"：输出是 rawvideo rgb24，没有 yuv420 的对齐要求


def ffmpeg_crop_scale(path, box, frame_hw=None, w=W, h=H, threads=0):
    """一条 ffmpeg 命令完成 解码 + 裁剪 + 缩放，返回 (T, h, w, 3) uint8（RGB）。
    frame_hw=(rows, cols) 时先按上游语义裁进画面；强烈建议总是传。"""
    x, y, bw, bh = clamp_box(box, frame_hw) if frame_hw else [int(round(v)) for v in box]
    vf = f"crop={bw}:{bh}:{x}:{y},scale={w}:{h}:flags=area"
    cmd = ["ffmpeg", "-v", "error", "-threads", str(threads), "-i", str(path),
           "-vf", vf, "-pix_fmt", "rgb24", "-f", "rawvideo", "-"]
    raw = subprocess.run(cmd, capture_output=True, check=True).stdout
    return np.frombuffer(raw, np.uint8).reshape(-1, h, w, 3)


TIMES = {}

class tick:
    """with tick("读视频"): ... —— 自动计时、累加、打印。"""
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        self.t = time.perf_counter()
        return self
    def __exit__(self, *exc):
        dt = time.perf_counter() - self.t
        TIMES[self.name] = TIMES.get(self.name, 0.0) + dt
        print(f"  ⏱ {self.name:<16}{dt:8.2f}s", flush=True)
        return False

def time_report():
    total = sum(TIMES.values()) or 1
    print("\n耗时明细" + "-" * 40)
    for k, v in sorted(TIMES.items(), key=lambda kv: -kv[1]):
        print(f"{k:<18}{v:8.2f}s  {v / total * 100:5.1f}%")
    print(f"{'合计':<18}{total:8.2f}s")

subject_dirs = sorted(p for p in DATASET_VIEW.glob("subject*") if (p / "vid.avi").exists())
subject_dirs = subject_dirs[:SUBSET] if SUBSET else subject_dirs
print(f"数据集 {DATASET_VIEW}: {len(subject_dirs)} 个 subject")
print(f"上游代码 {REPO}")


def dataset_fingerprint(dirs):
    """缓存指纹：只由"数据内容 + 预处理参数"决定，与 session / VM / 时间无关。
    换机器指纹不变 → 缓存自动复用；数据或参数一变指纹就变 → 不会用错缓存。"""
    h = hashlib.sha1()
    for d in dirs:
        v, g = d / "vid.avi", d / "ground_truth.txt"
        h.update(f"{d.name}|{v.stat().st_size}|{g.stat().st_size if g.exists() else 0}|".encode())
    h.update(f"|v{PREPROC_VERSION}|{W}x{H}|chunk{CHUNK}|coef{LARGE_COEF}|ffmpeg-area-clip".encode())
    return h.hexdigest()[:12]


FP = dataset_fingerprint(subject_dirs)
CACHE_DIR = CACHE_LOCAL / f"ubfc_{FP}"
print(f"数据集指纹 {FP}（subject 数 + 文件字节数 + 预处理参数）→ {CACHE_DIR}")


## 第 2 格 · 看数据，量读盘速度

先搞清楚瓶颈在哪：如果读盘就占了大头，那优化 Python 循环是白费力气。

`vid.avi` 是 **rawvideo** 编码（未压缩），所以一个 51 秒的 640×480 视频就 1.4 GB。


In [ ]:
# 第 2 格：数据规模 + 读盘吞吐
import cv2

sizes = [(d / "vid.avi").stat().st_size for d in subject_dirs]
one = float(np.median(sizes))
print(f"单个 vid.avi 中位 {one / 1e9:.2f} GB；{len(subject_dirs)} 个 = {one * len(subject_dirs) / 1e9:.1f} GB")

sample = subject_dirs[0] / "vid.avi"
cap = cv2.VideoCapture(str(sample))
fps = cap.get(cv2.CAP_PROP_FPS)
nfr = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fwh = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
cap.release()
print(f"样例 {sample.name}: {fwh[0]}x{fwh[1]} @ {fps:.1f}fps, {nfr} 帧 ≈ {nfr / fps:.0f} 秒")


def read_speed(path, frames=120):
    """顺序读 frames 帧，返回 MB/s（这是"能不能喂饱 GPU"的上限）。"""
    cap = cv2.VideoCapture(str(path))
    ok, _ = cap.read()
    cnt = nbytes = 0
    t0 = time.perf_counter()
    while ok and cnt < frames:
        ok, f = cap.read()
        if ok:
            cnt += 1
            nbytes += f.nbytes
    dt = time.perf_counter() - t0
    cap.release()
    return nbytes / dt / 1e6


with tick("测读盘"):
    mbps = read_speed(sample)
print(f"读盘吞吐 {mbps:.0f} MB/s  →  一个 {one / 1e9:.2f} GB 视频约需 {one / 1e6 / mbps:.0f} 秒")
print(f"{len(subject_dirs)} 个视频光读盘就 {len(subject_dirs) * one / 1e6 / mbps / 60:.1f} 分钟"
      f"（这就是 GPU 干等的那段时间）")


## 第 3 格 · 基线：上游的逐帧流程

对应 `dataset/data_loader/UBFCrPPGLoader.py:70-95`（读视频）+ `dataset/data_loader/BaseLoader.py:356-410`（裁剪缩放）。
逻辑一字不改，只把每一步拆开计时——慢在哪一步，一眼就看出来。

注意两个隐藏成本：
- `read_video` 把整段视频读成 `list` 再 `np.asarray`，中间有一份完整拷贝（1.4 GB → 2.8 GB）
- `crop_face_resize` 的 `np.zeros(...)` 没写 dtype，默认 **float64**，单视频 192 MB


In [ ]:
# 第 3 格：基线（上游逻辑复刻 + 分阶段计时）
def read_video(path):
    """= UBFCrPPGLoader.read_video：逐帧解码 + BGR→RGB，全部堆进内存"""
    cap = cv2.VideoCapture(str(path))
    ok, fr = cap.read()
    out = []
    while ok:
        out.append(np.asarray(cv2.cvtColor(np.array(fr), cv2.COLOR_BGR2RGB)))
        ok, fr = cap.read()
    cap.release()
    return np.asarray(out)


def crop_face_resize(frames, box):
    """= BaseLoader.crop_face_resize（静态框分支）"""
    x, y, bw, bh = box
    out = np.zeros((frames.shape[0], H, W, frames.shape[-1]))   # 上游默认 dtype=float64
    for i in range(frames.shape[0]):
        f = frames[i]
        f = f[max(y, 0):min(y + bh, f.shape[0]), max(x, 0):min(x + bw, f.shape[1])]
        out[i] = cv2.resize(f, (W, H), interpolation=cv2.INTER_AREA)
    return out


def standardized(data):
    """= BaseLoader.standardized_data（整段视频 z-score）"""
    d = data - np.mean(data)
    d = d / np.std(d)
    d[np.isnan(d)] = 0
    return d


with tick("读视频"):
    frames_slow = read_video(sample)
print(f"    帧数组 {frames_slow.shape} {frames_slow.dtype} = {frames_slow.nbytes / 1e6:.0f} MB")

with tick("人脸检测(1帧)"):
    box_slow = face_box(frames_slow[0])
print(f"    框 (x,y,w,h) = {box_slow}   上游 DO_DYNAMIC_DETECTION=False → 整段只检测这一帧")

with tick("裁剪+缩放"):
    crop_slow = crop_face_resize(frames_slow, box_slow)
print(f"    结果 {crop_slow.shape} {crop_slow.dtype} = {crop_slow.nbytes / 1e6:.0f} MB")

with tick("标准化+切块"):
    std_slow = standardized(crop_slow.copy())
    nclip = std_slow.shape[0] // CHUNK
    clips_slow = np.array([std_slow[i * CHUNK:(i + 1) * CHUNK] for i in range(nclip)])
print(f"    {nclip} 个 chunk, 形状 {clips_slow.shape}")

del frames_slow, crop_slow, std_slow   # 1.4 GB 的东西别留着占内存
time_report()


## 第 4 格 · 快版：ffmpeg 一次做完 decode + crop + scale

关键前提：**人脸框整段是常量**（上游 `DO_DYNAMIC_DETECTION: False`），所以 crop 参数是一个固定的四元组，
可以直接交给 ffmpeg 在 C 里逐帧做，Python 只负责收最后的 72×72 数据。

`flags=area` 对应 `cv2.INTER_AREA`。


In [ ]:
# 第 4 格：ffmpeg 一条命令完成 解码 + 裁剪 + 缩放（函数在第 1 格）
# 只解码第一帧做检测（对应上游"只检一次"），不用把整段视频读进来
with tick("首帧检测"):
    cap = cv2.VideoCapture(str(sample))
    ok, first = cap.read()
    cap.release()
    first = cv2.cvtColor(first, cv2.COLOR_BGR2RGB)      # 和上游一致：在 RGB 上检测
    box_fast = face_box(first)

with tick("ffmpeg 解码+裁剪"):
    crop_fast = ffmpeg_crop_scale(sample, box_fast, first.shape[:2])
print(f"    结果 {crop_fast.shape} {crop_fast.dtype} = {crop_fast.nbytes / 1e6:.0f} MB（基线的 1/8）")
time_report()


## 第 5 格 · 两条路差多少

裁剪区域必须相同才有可比性：这里**强制用同一个框**，只比较 resize 的实现差异。
`

- 像素平均差 < 1/255：说明只差插值舍入，可以放心换
- 如果差很大，说明框或颜色顺序（BGR/RGB）搞错了，先修那个再谈速度


In [ ]:
# 第 5 格：数值一致性
m = min(len(crop_slow), len(crop_fast))
a = crop_slow[:m].astype(np.float32)
b = crop_fast[:m].astype(np.float32)
d = np.abs(a - b)

print(f"帧数  基线={len(crop_slow)}  快版={len(crop_fast)}  对比 {m} 帧")
print(f"像素平均绝对差 {d.mean():.3f}/255   最大 {d.max():.1f}   完全相同像素 {(d == 0).mean() * 100:.1f}%")

a_std = standardized(a)
b_std = standardized(b)
print(f"标准化后平均绝对差 {np.abs(a_std - b_std).mean():.5f}（训练/推理看的是这个尺度）")

for i in (0, m // 2, m - 1):
    print(f"  第 {i:4d} 帧 平均差 {np.abs(a[i] - b[i]).mean():.3f}")


## 第 6 格 · 全量预处理 → 本地缓存

指纹只跟数据本身（subject 名、文件字节数）和预处理参数有关，**跟 session、VM、时间无关**：
换机器、换 session，只要数据和参数没变，指纹就不变，缓存自动复用。

顺带修上游一个坑：`BaseLoader.py:440` 的缓存名只有 `{filename}_input{count}.npy`，不含任何参数，
5 个实验共用一个 `CACHED_PATH` 会互相覆盖。

**为什么存 uint8**：裁剪完的像素本来就是 uint8，上游只是"先标准化再存"。
存原始像素 + 记录该视频的 mean/std，加载时用同一组数做 z-score，结果与上游一致，体积只有 1/4。


In [ ]:
# 第 6 格：全量预处理到本地缓存（CACHE_DIR 在第 1 格算好）
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def read_wave(path):
    """= UBFCrPPGLoader.read_wave：ground_truth.txt 第一行就是 BVP 波形"""
    with open(path) as f:
        return np.asarray([float(x) for x in f.read().split("\n")[0].split()])


def preprocess_one(d, cache_dir=CACHE_DIR):
    """一个视频：检测 1 帧 → ffmpeg 裁剪 → 存 uint8 chunk + 该视频的 mean/std。"""
    name = d.name
    cap = cv2.VideoCapture(str(d / "vid.avi"))
    ok, first = cap.read()
    cap.release()
    box = face_box(cv2.cvtColor(first, cv2.COLOR_BGR2RGB))
    arr = ffmpeg_crop_scale(d / "vid.avi", box, first.shape[:2])   # uint8 (T, 72, 72, 3)

    bvp = read_wave(d / "ground_truth.txt")
    bvp = (bvp - bvp.mean()) / (bvp.std() + 1e-8)               # LABEL_TYPE: Standardized

    n = min(len(arr), len(bvp)) // CHUNK
    for i in range(n):
        np.save(cache_dir / f"{name}_input{i}.npy", arr[i * CHUNK:(i + 1) * CHUNK])
        np.save(cache_dir / f"{name}_label{i}.npy", bvp[i * CHUNK:(i + 1) * CHUNK].astype(np.float32))

    stats = {"subject": name, "chunks": int(n), "frames": int(len(arr)), "box": box,
             "mean": float(arr.mean()), "std": float(arr.std())}
    (cache_dir / f"{name}_stats.json").write_text(json.dumps(stats, indent=1))
    return n


from concurrent.futures import ThreadPoolExecutor

# 为什么用线程不是进程：真正的重活（解码/裁剪/缩放）在 ffmpeg 子进程里，GIL 不是瓶颈；
# 2 个线程 = 2 条并发读盘，正好用满 Colab 标准档的 2 vCPU。
with tick(f"全量预处理({len(subject_dirs)}个)"):
    with ThreadPoolExecutor(max_workers=2) as ex:
        futures = [ex.submit(preprocess_one, d) for d in subject_dirs]
        for i, f in enumerate(futures, 1):
            f.result()
            print(f"   [{i}/{len(subject_dirs)}] 完成", end="\r", flush=True)

npy = sorted(CACHE_DIR.glob("*_input*.npy"))
mb = sum(p.stat().st_size for p in CACHE_DIR.glob("*.npy")) / 1e6
src_gb = sum((d / "vid.avi").stat().st_size for d in subject_dirs) / 1e9
print(f"\n缓存 {len(npy)} 个 chunk，共 {mb:.0f} MB"
      f"（原始视频 {src_gb:.1f} GB → 缓存 {mb / 1000:.2f} GB，缩小 {src_gb * 1000 / mb:.0f} 倍）")


## 第 7 格 · 缓存打包，上传 Drive

只有**第一次**（或数据集/参数变了）需要做这一步，做完之后新 VM 直接解包。

- 先打到本地盘 `/content/` 再复制到 Drive：直接往 FUSE 写 tar 又慢又容易断
- 同时写一个同名 `.json`：指纹、字节数、chunk 数、repo HEAD、生成时间——以后排查"这个 tar 是什么时候谁生成的"有据可查
- 文件名里带指纹，所以不同参数、不同子集的缓存**不会互相覆盖**


In [ ]:
# 第 7 格：打包 + 上传 Drive
import shutil

DRIVE_CACHE = Path("/content/drive/MyDrive/Colab_Projects/cache/preprocessed")
DRIVE_CACHE.mkdir(parents=True, exist_ok=True)
TAR = DRIVE_CACHE / f"ubfc_{FP}.tar.gz"
META = DRIVE_CACHE / f"ubfc_{FP}.json"
LOCAL_TAR = Path("/content") / TAR.name

with tick("打包 tar.gz"):
    subprocess.run(["tar", "-czf", str(LOCAL_TAR), "-C", str(CACHE_DIR), "."], check=True)
size = LOCAL_TAR.stat().st_size
print(f"    tar {size / 1e6:.0f} MB")

with tick("上传 Drive"):
    shutil.copy2(LOCAL_TAR, TAR)

META.write_text(json.dumps({
    "fingerprint": FP,
    "tar": TAR.name,
    "bytes": size,
    "chunks": len(list(CACHE_DIR.glob("*_input*.npy"))),
    "dtype": "uint8",
    "chunk_length": CHUNK,
    "resize": [W, H],
    "large_box_coef": LARGE_COEF,
    "subjects": [d.name for d in subject_dirs],
    "repo_head": subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"],
                                capture_output=True, text=True).stdout.strip(),
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
}, ensure_ascii=False, indent=1), encoding="utf-8")

LOCAL_TAR.unlink()          # 本地不留副本
print(f"已上传 {TAR}（{size / 1e6:.0f} MB）+ {META.name}")


## 第 8 格 · 新 VM 解包（有 tar 就跳过第 6 格）

顺序是：先跑第 1 格（得到 `FP`）→ 跑这一格 → 如果 Drive 上没有同指纹的 tar，再回去跑第 6、7 格。
解包约几十秒，而重新预处理 + 从 Drive 读 60 GB 原始视频要好几分钟到几十分钟。


In [ ]:
# 第 8 格：缓存复用（解包）—— 新 VM 上跑完第 1 格就能直接跑这格
import shutil

DRIVE_CACHE = Path("/content/drive/MyDrive/Colab_Projects/cache/preprocessed")
TAR = DRIVE_CACHE / f"ubfc_{FP}.tar.gz"
META = DRIVE_CACHE / f"ubfc_{FP}.json"

have = len(list(CACHE_DIR.glob("*_input*.npy"))) if CACHE_DIR.exists() else 0
if have:
    print(f"本地已有缓存：{have} 个 chunk，跳过")
elif TAR.exists():
    LOCAL_TAR = Path("/content") / TAR.name
    with tick("下载 tar"):
        shutil.copy2(TAR, LOCAL_TAR)
    with tick("解包"):
        CACHE_DIR.mkdir(parents=True, exist_ok=True)
        subprocess.run(["tar", "-xzf", str(LOCAL_TAR), "-C", str(CACHE_DIR)], check=True)
    LOCAL_TAR.unlink()
    meta = json.loads(META.read_text()) if META.exists() else {}
    print(f"解包完成：{len(list(CACHE_DIR.glob('*_input*.npy')))} 个 chunk"
          f"（生成于 {meta.get('created_at', '?')}，repo {meta.get('repo_head', '?')[:8]}）")
else:
    print(f"Drive 上没有 {TAR.name} → 回去跑第 6、7 格做第一次预处理")


## 第 9 格 · 边产边推：预处理与 GPU 推理重叠

上游是**串行**的：42 个视频全预处理完 → 才开始推理，所以前面 GPU 一直 0%。

这里用两根线程：一根持续做"解码→裁剪→切块"，另一根只要有 chunk 就喂模型。
队列 `maxsize=4` 是背压——预处理再快也不会把内存吃爆。

队列里只放 **uint8 像素 + 该视频的 mean/std**（第 6 格定的格式），
消费端再做 z-score，等价于上游"先标准化整段再切块"。


In [ ]:
# 第 9 格：流水线（生产者-消费者）
import queue
import threading

MIN_HZ, MAX_HZ = 0.75, 2.5      # 0.75Hz=45bpm，2.5Hz=150bpm，rPPG 常用频带
N_VIDEOS = 3                    # 先跑 3 个视频看流水线效果，确认了再放大


def load_model():
    """载入上游 EfficientPhys（PURE 预训练权重），一行不改上游代码。"""
    import importlib

    sys.path.insert(0, str(REPO))
    last = None
    for pkg in ("neural_methods", "Neural_methods"):     # 目录名实际是小写
        try:
            mod = importlib.import_module(f"{pkg}.model.EfficientPhys")
            break
        except ModuleNotFoundError as e:
            last = e
    else:
        raise last

    m = mod.EfficientPhys(
        in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3,
        dropout_rate1=0.25, dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128,
        frame_depth=FRAME_DEPTH, img_size=H, channel="raw",
    ).to(DEVICE)

    sd = torch.load(REPO / "final_model_release/PURE_EfficientPhys.pth",
                    map_location="cpu")
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}   # DataParallel 前缀
    m.load_state_dict(sd)
    return m.eval()


def hr_from_signal(sig, fs=30.0, lo=MIN_HZ, hi=MAX_HZ):
    """峰值法估心率：去均值 → Hann 窗 → rFFT → 频带内取峰 → bpm"""
    x = np.asarray(sig, dtype=np.float64)
    x = x - x.mean()
    x = x * np.hanning(len(x))
    freqs = np.fft.rfftfreq(len(x), d=1.0 / fs)
    spec = np.abs(np.fft.rfft(x))
    band = (freqs >= lo) & (freqs <= hi)
    if not band.any() or spec[band].max() <= 0:
        return float("nan")
    return float(freqs[band][np.argmax(spec[band])] * 60.0)


model = load_model()
print(f"模型已载入：{sum(p.numel() for p in model.parameters())/1e6:.2f} M 参数，device={DEVICE}")

q = queue.Queue(maxsize=4)
results = []


def producer(dirs):
    for d in dirs:
        cap = cv2.VideoCapture(str(d / "vid.avi"))
        ok, first = cap.read()
        cap.release()
        box = face_box(cv2.cvtColor(first, cv2.COLOR_BGR2RGB))
        arr = ffmpeg_crop_scale(d / "vid.avi", box, first.shape[:2])   # uint8 (T, 72, 72, 3)
        bvp = read_wave(d / "ground_truth.txt")
        bvp = (bvp - bvp.mean()) / (bvp.std() + 1e-8)
        n = min(len(arr), len(bvp)) // CHUNK
        mu, sd = float(arr.mean()), float(arr.std())
        for i in range(n):
            q.put((d.name, i, arr[i * CHUNK:(i + 1) * CHUNK], mu, sd,
                   bvp[i * CHUNK:(i + 1) * CHUNK]))
    q.put(None)


t0 = time.time()
th = threading.Thread(target=producer, args=(subject_dirs[:N_VIDEOS],), daemon=True)
th.start()

gpu_busy = 0.0
while True:
    item = q.get()
    if item is None:
        break
    name, idx, chunk, mu, sd, label = item

    # 消费端标准化：用该视频的 mean/std，与上游"整段标准化"等价
    x = (chunk.astype(np.float32) - mu) / (sd + 1e-8)
    x = torch.from_numpy(x).permute(0, 3, 1, 2).to(DEVICE)   # (T,3,H,W)

    with torch.no_grad():
        _t = time.perf_counter()
        pred = model(x).squeeze(-1).cpu().numpy()             # (T-1,)
        gpu_busy += time.perf_counter() - _t

    hr_pred = hr_from_signal(pred)
    hr_true = hr_from_signal(label)
    results.append({"video": name, "chunk": idx, "pred": hr_pred, "true": hr_true})
    el = time.time() - t0
    print(f"[{el:5.1f}s] {name} chunk{idx} 预测 {hr_pred:5.1f} bpm / 真值 {hr_true:5.1f}"
          f"  队列 {q.qsize()}", flush=True)

el = time.time() - t0
print(f"\n{N_VIDEOS} 个视频 / {len(results)} 个 chunk 用时 {el:.1f}s"
      f"（{el / max(len(results), 1):.2f}s per chunk）")
print(f"其中模型推理 {gpu_busy:.1f}s → GPU 占用估计 {gpu_busy / max(el, 1e-9) * 100:.0f}%")


## 第 10 格 · 汇总

只统计 ±3 bpm 命中率这类粗口径，具体论文口径以第 0 格导入的 `experiments` 为准。


In [ ]:
# 第 10 格：指标
if results:
    pred = np.array([r["pred"] for r in results])
    true = np.array([r["true"] for r in results])
    within = np.abs(pred - true) <= 3     # ±3 bpm 内算对，FFT 评测常用的粗口径
    print(f"chunk 数 {len(results)}")
    print(f"MAE      {np.abs(pred - true).mean():6.2f} bpm")
    print(f"RMSE     {np.sqrt(((pred - true) ** 2).mean()):6.2f} bpm")
    print(f"±3bpm 命中率 {within.mean() * 100:5.1f}%")
    print(f"预测心率范围 {pred.min():.0f}-{pred.max():.0f} bpm；真值 {true.min():.0f}-{true.max():.0f} bpm")

    TIMES.clear()
    print()
    print("对照：上游 42 个视频预处理耗时（本地实测 1.2s/视频，Drive 读取另算）")
else:
    print("第 9 格没产出结果，先回去看报错")
